In [ ]:
# =============================================================================
# Project 2 – PageRank Algorithm  |  Zane Mullinax 
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

print("=" * 80)
print("Project 2")
print("=" * 80)
print()

# -----------------------------------------------------------------------------
# PART 1: Construct the transition matrix A for Figure 1
# -----------------------------------------------------------------------------
print("PART 1: Constructing the transition matrix A for Figure 1")
print("-" * 80)

# Convention: A[i, j] = P( go FROM page j TO page i )

# From page 1:
#   2 outgoing links -> one to page 2, one to page 3
#   Probabilities: P(1→1) = 0, P(1→2) = 1/2, P(1→3) = 1/2
col1 = np.array([0.0, 1.0, 1.0], dtype=float) / 2.0

# From page 2:
#   3 outgoing links -> one to page 1, two to page 3
#   Probabilities: P(2→1) = 1/3, P(2→2) = 0, P(2→3) = 2/3
col2 = np.array([1.0, 0.0, 2.0], dtype=float) / 3.0

# From page 3:
#   1 outgoing link  -> to page 1 only
#   Probabilities: P(3→1) = 1, P(3→2) = 0, P(3→3) = 0
col3 = np.array([1.0, 0.0, 0.0], dtype=float)

# Build the transition matrix A by stacking columns
A = np.column_stack((col1, col2, col3))

print("Transition matrix A:")
print(A)
print(f"\nShape of A: {A.shape}")
print("Reminder: A[i, j] = P(from page j to page i)")
print()


# -----------------------------------------------------------------------------
# PART 2: Validate that A is a proper transition matrix
# -----------------------------------------------------------------------------
print("PART 2: Validating the transition matrix")
print("-" * 80)

# Check nonnegativity
all_nonnegative = np.all(A >= 0)

# Check column sums (must each be 1 for a column-stochastic matrix)
column_sums = A.sum(axis=0)
columns_sum_to_one = np.allclose(column_sums, 1.0)

print("Validation checks for A:")
print(f"  All entries >= 0?           {all_nonnegative}")
print(f"  Column sums (should be 1):  {column_sums}")
print(f"  All columns sum to 1?       {columns_sum_to_one}")
print()

if not (all_nonnegative and columns_sum_to_one):
    print("WARNING: A is NOT a valid transition matrix. Check the definition.")
else:
    print("A is a valid column-stochastic (transition) matrix.")
print()


# -----------------------------------------------------------------------------
# PART 3: PageRank iteration x_{n+1} = A x_n
# -----------------------------------------------------------------------------

print("PART 3: Running the PageRank iteration")
print("-" * 80)

# Number of iterations (you can increase N if you want)
N = 50

# Starting probability vector x0 (uniform distribution over 3 pages)
x0 = np.full(3, 1.0 / 3.0, dtype=float)

# Storage for the whole sequence: 3 rows (pages), N+1 columns (0..N)
history = np.zeros((3, N + 1), dtype=float)

# Store x0 as the first column
history[:, 0] = x0

# Function to test if a vector is a probability vector

def is_probability_vector(x, tol=1e-10):
    nonnegative = np.all(x >= -tol)
    sums_to_one = abs(float(x.sum()) - 1.0) <= tol
    return nonnegative and sums_to_one

print(f"Initial vector x0: {x0}")
print(f"Is x0 a probability vector? {is_probability_vector(x0)}")
print()

# Run the iterations: x_{n+1} = A x_n
for n in range(N):
    history[:, n + 1] = A @ history[:, n]

# Print a few sample iterations
print("Selected iterations (columns are probability distributions over pages):")
for n in [0, 1, 2, 5, 10, N]:
    print(f"  x_{n:2d} = {history[:, n]}")
print()

# Verify that each x_n remains a probability vector
valid_flags = [is_probability_vector(history[:, k]) for k in range(N + 1)]
print("Does each x_n remain a probability vector?")
print(f"  {valid_flags}")
print(f"All True? {all(valid_flags)}")
print()


# -----------------------------------------------------------------------------
# PART 4: Limiting distribution x_infinity and fixed-point check
# -----------------------------------------------------------------------------
print("PART 4: Limiting distribution x_infinity and fixed-point property")
print("-" * 80)

# Approximate x_infinity by the last iterate
x_infinity = history[:, -1]

print(f"x_infinity (approx) = x_{N} = {x_infinity}")
print(f"Sum of x_infinity: {x_infinity.sum():.10f}")
print(f"Is x_infinity a probability vector? {is_probability_vector(x_infinity)}")
print()

# Check fixed-point: A x_infinity ≈ x_infinity
Ax_inf = A @ x_infinity
fixed_point_error = np.linalg.norm(Ax_inf - x_infinity)

print("Fixed-point check:")
print(f"  A @ x_infinity = {Ax_inf}")
print(f"  ||A x_infinity - x_infinity|| = {fixed_point_error:.10e}")
print()


# -----------------------------------------------------------------------------
# PART 5: Convergence plot – ||x_infinity - x_n|| versus n
# -----------------------------------------------------------------------------
print("PART 5: Convergence behavior")
print("-" * 80)

errors = np.zeros(N + 1, dtype=float)
for step in range(N + 1):
    errors[step] = np.linalg.norm(x_infinity - history[:, step])

print("First few distances ||x_infinity - x_n||:")
for n in [0, 1, 2, 5, 10, N]:
    print(f"  n = {n:2d}, distance = {errors[n]:.10e}")
print()

plt.figure()
plt.plot(range(N + 1), errors, marker="o")
plt.xlabel("Iteration n")
plt.ylabel(r"$\| x_\infty - x_n \|_2$")
plt.title("Convergence of PageRank Iteration")
plt.grid(True)
plt.show()


# -----------------------------------------------------------------------------
# PART 6: Eigenvalues and eigenvectors of A
# -----------------------------------------------------------------------------
print("PART 6: Eigenvalues and eigenvectors")
print("-" * 80)

eigvals, eigvecs = np.linalg.eig(A)

print("Eigenvalues of A:")
print(eigvals)
print()

print("Eigenvectors (columns of eigvecs) – NOT normalized:")
print(eigvecs)
print()

# Find the eigenvector associated with eigenvalue 1 (or closest to 1)
idx = np.argmin(np.abs(eigvals - 1.0))
v_stationary = np.real(eigvecs[:, idx])

# Normalize this eigenvector to sum to 1, to compare with x_infinity
v_stationary_normalized = v_stationary / v_stationary.sum()

print("Eigenvector associated with eigenvalue ~1 (before normalization):")
print(v_stationary)
print()

print("Stationary eigenvector normalized to sum to 1:")
print(v_stationary_normalized)
print(f"Sum: {v_stationary_normalized.sum():.10f}")
print()

print("Distance between x_infinity and normalized eigenvector:")
print(
    "  ||x_infinity - v_stationary_normalized|| = "
    f"{np.linalg.norm(x_infinity - v_stationary_normalized):.10e}"
)
print()


# -----------------------------------------------------------------------------
# PART 7: Multiple starting vectors and independence of starting point
# -----------------------------------------------------------------------------
print("PART 7: Testing different starting vectors")
print("-" * 80)

def run_pagerank(A, x_start, N=50):
    """Run N iterations of x_{n+1} = A x_n starting from x_start."""
    current = x_start.astype(float).copy()
    for _ in range(N):
        current = A @ current
    return current

starting_vectors = [
    ("Uniform (1/3, 1/3, 1/3)", np.full(3, 1.0 / 3.0)),
    ("Page 1 only",             np.array([1.0, 0.0, 0.0])),
    ("Page 2 only",             np.array([0.0, 1.0, 0.0])),
    ("Page 3 only",             np.array([0.0, 0.0, 1.0])),
    ("Biased (0.8, 0.1, 0.1)",  np.array([0.8, 0.1, 0.1])),
]

for label, x_start in starting_vectors:
    x_final = run_pagerank(A, x_start, N=N)
    err = np.linalg.norm(x_final - x_infinity)

    print(label + ":")
    print(f"  start: {x_start}")
    print(f"  final: {x_final}")
    print(f"  ||x_final - x_infinity|| = {err:.10e}")
    print()

print("Observation: All reasonable starting probability vectors converge")
print("to the same limiting distribution x_infinity (up to numerical error).")
print()
